In [ ]:
import os
from dotenv import load_dotenv
from pyprojroot import here
from typing import List
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from pprint import pprint
from openai import OpenAI # Import the OpenAI class
import pandas as pd
import sqlite3 #in built database
import json
import re
from langchain.chains import create_sql_query_chain
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate



In [ ]:


# os.environ["OPENAI_API_TYPE"] = "azure"
# os.environ["OPENAI_API_KEY"] = ""
# os.environ["OPENAI_API_BASE"] = "https://fabric-coe.openai.azure.com/"
# os.environ["OPENAI_API_VERSION"] = "2024-05-01-preview"
# os.environ["OPENAI_DEPLOYMENT_NAME"] = "gpt-4o"

# OpenAI.api_key = os.environ.get('OPENAI_API_KEY') # Access the api_key using the class name
# client=OpenAI()
#gpt-4o-mini

In [ ]:
# print(os.environ.get("OPENAI_API_KEY"))  # Should print your API key
# print(os.environ.get("OPENAI_API_BASE"))  # Should print your Azure API base URL

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_openai import AzureChatOpenAI

# Initialize Azure OpenAI LLM
llm = AzureChatOpenAI(
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    azure_endpoint=os.getenv("OPENAI_API_BASE"),  # Use 'azure_endpoint' instead of 'openai_api_base'
    deployment_name=os.getenv("OPENAI_DEPLOYMENT_NAME"),  # Use deployment name instead of model name
    temperature=0
)

Reading data from excel

In [ ]:
shell_dim_df = pd.read_csv(r"/content/Shell__dim_station__preview_.csv")
shell_fact_df = pd.read_csv(r"/content/Shell__fact_station_day_product__preview_.csv")







Setup Database

In [ ]:
database_file = "shell_newdemo.db" #it is stored in sqlite

In [ ]:
conn = sqlite3.connect("shell_newdemo.db")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

Creating tables schema in database

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_station (
    station_id TEXT PRIMARY KEY,       -- Unique ID for each fuel station (e.g., BLR-001)
    station_name TEXT NOT NULL,        -- Display/station name
    city TEXT NOT NULL,                -- City where the station is located
    cluster TEXT,                      -- Business/operational cluster grouping
    latitude REAL NOT NULL,            -- Latitude coordinate
    longitude REAL NOT NULL,           -- Longitude coordinate
    opened_year INTEGER,               -- Opening year of the station
    has_ev_charger INTEGER,            -- EV charger availability (0 = No, 1 = Yes)
    cstore_size_sqft INTEGER           -- Convenience store area in square feet
);
""")


In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_station (
    date TEXT NOT NULL,                                      -- Date of transaction (YYYY-MM-DD)
    station_id TEXT NOT NULL,                                -- Unique station ID (FK to dim_station)
    city TEXT NOT NULL,                                      -- City for easy filtering
    product_family TEXT NOT NULL,                            -- Fuel product type (Petrol, Diesel, Premium)

    shell_price_inr_per_liter REAL NOT NULL,                 -- Shell’s selling price per liter
    comp_min_price_inr_per_liter_within_3km REAL,            -- Minimum competitor price nearby
    price_gap_inr_per_liter REAL,                            -- Difference between Shell price & comp min price

    liters_sold INTEGER NOT NULL,                            -- Number of liters sold
    revenue_inr REAL NOT NULL,                               -- Total fuel revenue (INR)
    gross_margin_inr REAL NOT NULL,                          -- Gross margin earned (INR)

    downtime_minutes INTEGER DEFAULT 0,                      -- Pump downtime minutes
    stockout_flag INTEGER DEFAULT 0,                         -- 1 if stockout happened, else 0
    promo_active INTEGER DEFAULT 0,                          -- 1 if promo was running, else 0

    competitors_within_3km INTEGER,                          -- Competitor station count near site

    weather_heat_index REAL,                                 -- Temperature indicator (°C-like)
    rainfall_mm REAL,                                        -- Rainfall measure for demand impact
    holiday_flag INTEGER DEFAULT 0,                          -- 1 if public holiday, else 0

    footfall_estimate INTEGER,                               -- Estimated total footfall count
    cstore_transactions INTEGER,                             -- Number of C-store purchases
    cstore_revenue_inr REAL,                                 -- Revenue from C-store

    loyalty_signups INTEGER,                                 -- New loyalty signups
    ev_charger_sessions INTEGER,                             -- EV charging sessions count

    PRIMARY KEY (date, station_id, product_family),          -- Composite key ensures uniqueness
    FOREIGN KEY (station_id) REFERENCES dim_station(station_id)  -- Referential integrity
);
""")


In [ ]:
db = SQLDatabase.from_uri(f"sqlite:///{database_file}")

In [ ]:
print(db.dialect)
print(db.get_usable_table_names())

In [ ]:
shell_fact_df["date"] = pd.to_datetime(shell_fact_df["date"], format="%d-%m-%Y").dt.strftime("%Y-%m-%d")


Storing data from excel to tables created above

In [ ]:
# 3. Write all DataFrames to their respective tables
shell_dim_df.to_sql("dim_station", conn, if_exists="replace", index=False)
shell_fact_df.to_sql("fact_station", conn, if_exists="replace", index=False)



In [ ]:
results = cursor.execute("SELECT count(*) FROM fact_station").fetchall()
print(results)

In [ ]:
results = cursor.execute("SELECT count(*) FROM dim_station").fetchall()
print(results)

Defining columns description

In [ ]:


# Step 1: Define Column Descriptions
dim_station_column_descriptions = {
    "station_id": "Unique identifier for each Shell fuel station (e.g., BLR-001).",
    "station_name": "Display name of the station used for business reporting.",
    "city": "Metro/urban location where the station operates.",
    "cluster": "Operational grouping of stations for performance management.",
    "latitude": "Geographic latitude of the station, useful for mapping and routing.",
    "longitude": "Geographic longitude of the station.",
    "opened_year": "Year the station started operations, helpful for lifecycle and maturity analysis.",
    "has_ev_charger": "Indicates if the station supports EV charging (1 = Yes, 0 = No).",
    "cstore_size_sqft": "Size of the Shell convenience store in square feet, used for revenue potential analysis."
}






fact_station_column_descriptions = {
    "date": "Daily date for transaction reporting in YYYY-MM-DD format.",
    "station_id": "Unique identifier linking to dim_station for each Shell site.",
    "city": "City-level filter for localized performance analytics.",
    "product_family": "Fuel type sold: Petrol, Diesel, or Premium.",

    "shell_price_inr_per_liter": "Shell retail selling price per liter in INR.",
    "comp_min_price_inr_per_liter_within_3km": "Minimum competitor price detected within a 3km radius.",
    "price_gap_inr_per_liter": "Pricing difference: Shell price minus competitor minimum price (positive → Shell is pricier).",

    "liters_sold": "Total fuel volume sold for the product that day (in liters).",
    "revenue_inr": "Total revenue generated from fuel sales in INR.",
    "gross_margin_inr": "Gross margin earned on fuel sales for the day in INR.",

    "downtime_minutes": "Pump/equipment downtime duration impacting sales opportunity.",
    "stockout_flag": "Indicates if a product was unavailable for any duration (1 = Stockout, 0 = Normal).",
    "promo_active": "Indicates whether a discount/offer/promotion was active (1 = Yes, 0 = No).",

    "competitors_within_3km": "Number of competitor stations competing for the same catchment area.",

    "weather_heat_index": "Approximate temperature/humidity index that influences fuel demand.",
    "rainfall_mm": "Rainfall amount that may impact footfall and demand fluctuations.",
    "holiday_flag": "Marks national/major holidays that drive demand changes (1 = Holiday).",

    "footfall_estimate": "Estimated number of customers visiting the station on that day.",
    "cstore_transactions": "Number of completed transactions in the convenience store.",
    "cstore_revenue_inr": "Revenue generated from non-fuel C-store sales.",

    "loyalty_signups": "Number of new enrollments into Shell loyalty programmes.",
    "ev_charger_sessions": "Count of EV charging sessions (if facility exists)."
}




Defining relationship between tables

In [ ]:
# Step 2: Define Relationships for Shell Retail DB
relationships = {
    "fact_station_day_product": {
        "station_id": {
            "references": {
                "table": "dim_station",
                "column": "station_id"
            },
            "relationship_type": "many_to_one"
        }
    }
}


In [ ]:
conn.commit()
#conn.close()

In [ ]:
from langchain_community.utilities import SQLDatabase
from pyprojroot import here
import warnings
warnings.filterwarnings("ignore")

In [ ]:
db = SQLDatabase.from_uri(f"sqlite:///{database_file}")

NEW VERSION OF CODE

In [ ]:
%pip install --upgrade --quiet  langchain langchain-community langchain-openai

In [ ]:
print(db.run("SELECT * FROM dim_station LIMIT 10;"))

In [ ]:
table_info_combined = (
    "dim_station(station_id, station_name, city, cluster, latitude, longitude, opened_year, has_ev_charger, cstore_size_sqft)\n"
    "fact_station(date, station_id, city, product_family, shell_price_inr_per_liter, "
    "comp_min_price_inr_per_liter_within_3km, price_gap_inr_per_liter, liters_sold, revenue_inr, gross_margin_inr, "
    "downtime_minutes, stockout_flag, promo_active, competitors_within_3km, weather_heat_index, rainfall_mm, "
    "holiday_flag, footfall_estimate, cstore_transactions, cstore_revenue_inr, loyalty_signups, ev_charger_sessions)\n"
)


In [ ]:
db.get_context()["table_info"] = table_info_combined
table_info = db.get_context().get("table_info", "")
top_k=50


In [ ]:
question = "Why are Bangalore clusters showing lower gross margin than Chennai?"

In [ ]:
# --- Shell: Text-to-SQL user prompt ---
user_prompt = f"""
You are an expert data analyst for Shell Retail (India) working with SQLite.

Here is the user question:
{question}

Here are the details of the dataset:

table_name1 = 'dim_station' and column description = {dim_station_column_descriptions}
table_name2 = 'fact_station' and column description = {fact_station_column_descriptions}

## Relationships:
Use ONLY the relationships provided here (many-to-one):
{relationships}
- fact_station_day_product.station_id → dim_station.station_id

## Consider the relevant table info:
{table_info_combined}

Strict Rules:
- Use ONLY the tables/columns listed above. Do not invent tables or columns.
- Enforce ONLY the relationship defined above when joining.
- Dates are TEXT in 'YYYY-MM-DD'. Use SQLite date helpers when needed (e.g., DATE('now','start of month')).
- Return a single valid SQLite SELECT query. No comments, no explanations, no markdown fences.
- Use camelcase fo city names
- Prefer explicit column lists; avoid SELECT *.
- If the user asks for “Top/Bottom N,” use ORDER BY and LIMIT.
- Round off all the values upto two decimal places.
- Unless the user specifies otherwise, LIMIT results to {top_k}.
"""



response = llm.invoke([
    ("system", "You are a Shell Retail SQLite expert. Given an input question and schema, return ONLY a syntactically correct SQLite SELECT query that follows the provided relationships and rules."),
    ("human", user_prompt)
])

sql_query = response.content  # this is the generated SQL


In [ ]:
sql_query=response.content
sql_query =sql_query.replace('sql','').replace('`','')


In [ ]:
print(sql_query)

In [ ]:
result = db.run(sql_query)

In [ ]:
print(result)

**SQL Agent Insights**

In [ ]:
insight_prompt = f"""
You are a senior Shell Retail business performance analyst.
Your job is to interpret SQL results and explain *why* performance is the way it is,
connecting data patterns to pricing, competition, sales leakage, conversion, and operational excellence.

Context:
- Audience: Shell Retail Operations and Cluster Leadership Teams.
- These insights inform business reviews and decision boards.
- You have deep knowledge of station operations, pricing dynamics, and KPIs.

Inputs:
• Business Question:
{question}

• SQL Output (tabular data):
{result}

Your Task:
Generate an analytical summary that answers the question comprehensively.
Go beyond reporting numbers — explain *why* these numbers look this way,
which operational or market factors may be influencing them,
and what specific actions Shell teams should consider next.

Guidelines:
1. Start with a brief snapshot of the key trend or variance.
2. Diagnose likely drivers across these pillars:
   - Pricing & Competitiveness (price gap vs. nearby competition)
   - Customer Demand (footfall, conversion, loyalty signups)
   - Operational Efficiency (downtime, stockouts, EV charger usage)
   - Promotional Effectiveness (active promos vs. uplift)
   - Network Health (cluster performance, city-wise variance)
3. Quantify differences or gaps where possible.
4. End with 2–3 concrete recommendations or next steps:
   - Pricing adjustments
   - Promo targeting
   - Stockout reduction
   - EV charger optimization
   - Uptime improvements

Rules:
- Be concise and structured — 3–6 bullet points total.
- Use the data in the result table; never speculate beyond it.
- Avoid technical SQL language or metadata.
- No generic “monitor further” statements; give specific, actionable insights.
- Assume readers know Shell’s business context but not raw numbers.

Output Format:
📊 **What’s happening**
📉 **Why it’s happening (drivers & root causes)**
🎯 **Recommended business actions**
"""
response = llm.invoke(
    [
        ("system", "You are a data-driven business analyst skilled at diagnostic storytelling for Shell Retail leadership."),
        ("human", insight_prompt)
    ]
)


In [ ]:
print(response.content)